# DataGuide 재무데이터 분석 (korea_fs_data_from_DG)

`dataguide_fs_analyzer_v1.py` 사용 예시. 모듈은 `Korea_Market/analysis/` 등 노트북과 같은 폴더 또는 sys.path 상에 두면 됩니다.

표시 단위: 억원 (`unit=1e5`, 천원→억원). 분기 인덱스는 `Period('Q')` 로 정규화 (회계분기말 영업일 문제 해소).

In [16]:
# ==========================================================
# PART 1: 환경 + Control Panel + DB
# ==========================================================
import sys
from pathlib import Path
import pandas as pd

def add_repo_path():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'DATA').exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            return parent
    raise FileNotFoundError('DATA 폴더를 찾을 수 없습니다')

PROJECT_ROOT = add_repo_path()
sys.path.insert(0, str(Path.cwd()))

from DATA import config
import dataguide_fs_analyzer_v1 as A

engine = config.get_engine(config.get_db_info())

# ---- Control Panel ----
ASOF    = None      # 기준 분기. 예: '2026Q2'. None → 최근 완전 적재 분기 자동 선택
MARKET  = None      # 'KS' / 'KQ' / None(전체)
TOP_N   = 200        # 스크리너 상위 N개
UNIT    = 1e5       # 천원 → 억원

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. 수집된 재무항목 목록 (키워드 검색)

In [17]:
A.search_items(engine)                 # 전체
# A.search_items(engine, '이익')       # 키워드 부분일치
# A.search_items(engine, '차입')

,alias,item_code,indicator,sj_div,n_ticker,min_date,max_date
0,매입채무,M000902006,매입채무(천원),BS,1521,2009-12-30,2026-06-30
1,단기금융상품,M001113350,단기금융상품(금융기관예치금)(천원),BS,1560,2009-12-30,2026-06-30
2,,M001113970,선급비용(천원),BS,1579,2009-12-30,2026-06-30
3,단기차입금,M001121700,단기차입금(*)(천원),BS,1522,2009-12-30,2026-06-30
4,리스부채,M001122020,(금융)리스부채(천원),BS,1508,2009-12-30,2026-06-30
5,,M001122140,미지급금(*)(천원),BS,1582,2009-12-30,2026-06-30
6,,M001122290,미지급비용(천원),BS,1579,2009-12-30,2026-06-30
7,,M001122561,계약부채(천원),BS,722,2017-12-28,2026-06-30
8,,M001122580,선수금(*)(천원),BS,1553,2009-12-30,2026-06-30
9,비지배지분,M001130640,비지배주주지분(천원),BS,1171,2009-12-30,2026-06-30


## 2. 종목별 시계열 (행=분기, 열=항목)
항목은 별칭(`A.ITEM_ALIAS` 참조) / item_code / 항목명 부분일치 모두 가능.

In [18]:
ts = A.get_ts(engine, 'A278470', ['매출액', '매출총이익', '영업이익', '당기순이익', '자본', '영업현금흐름'], start='2023-01-01', unit=UNIT)
print(ts.attrs)
ts

{'ticker': 'A278470', 'company_name': '에이피알'}


,매출액,매출총이익,영업이익,당기순이익,자본,영업현금흐름
q,,,,,,
2023Q1,"1,221.79",906.74,231.89,202.96,"1,291.03",340.18
2023Q2,"1,276.72",976.40,247.85,187.66,"1,486.15",266.15
2023Q3,"1,219.39",905.08,218.60,183.78,"1,686.75",2.04
2023Q4,"1,520.19","1,166.26",343.60,241.05,"1,969.49",470.04
2024Q1,"1,489.28","1,153.34",277.65,240.93,"2,974.76",194.63
2024Q2,"1,554.94","1,188.00",280.11,240.99,"3,087.26",82.80
2024Q3,"1,741.17","1,307.67",272.43,160.07,"2,815.27",200.06
2024Q4,"2,442.15","1,786.99",396.86,433.91,"3,235.24",313.74
2025Q1,"2,660.33","2,008.65",545.68,499.41,"3,455.23",535.10


In [19]:
# 시계열 적재 상태 확인: 결측 분기 체크
ts.isna().sum()

매출액       0
매출총이익     0
영업이익      0
당기순이익     0
자본        0
영업현금흐름    0
dtype: int64

## 3. YoY 성장률 상위 N개

In [20]:
A.yoy_screen(engine, '매출액', n=TOP_N, asof=ASOF, market=MARKET, unit=UNIT)

,ticker,company_name,base_q,t_q,매출액(t-4),매출액(t),growth_%
0,A255440,야스,2025Q2,2026Q2,61.48,872.20,"1,318.79"
1,A402340,SK스퀘어,2025Q2,2026Q2,"18,883.33","196,097.74",938.47
2,A000040,KR모터스,2025Q2,2026Q2,45.36,263.48,480.81
3,A080220,제주반도체,2025Q2,2026Q2,510.67,"2,899.38",467.76
4,A039200,오스코텍,2025Q2,2026Q2,100.17,526.49,425.61
...,...,...,...,...,...,...,...
195,A246710,티앤알바이오팹,2025Q2,2026Q2,65.36,95.45,46.03
196,A044820,코스맥스비티아이,2025Q2,2026Q2,"1,637.26","2,389.77",45.96
197,A005690,파미셀,2025Q2,2026Q2,267.63,389.80,45.65
198,A317330,덕산테코피아,2025Q2,2026Q2,289.67,421.82,45.62


In [21]:
A.yoy_screen(engine, '영업이익', n=TOP_N, asof=ASOF, market=MARKET, unit=UNIT, min_base=5e5)   # 기준 영업이익 ≥ 5억원


,ticker,company_name,base_q,t_q,영업이익(t-4),영업이익(t),growth_%
0,A016450,한세예스24홀딩스,2025Q2,2026Q2,5.40,403.93,"7,376.37"
1,A353200,대덕전자,2025Q2,2026Q2,18.66,702.76,"3,665.96"
2,A080220,제주반도체,2025Q2,2026Q2,43.38,"1,218.71","2,709.48"
3,A005950,이수화학,2025Q2,2026Q2,39.09,965.59,"2,370.21"
4,A011070,LG이노텍,2025Q2,2026Q2,113.92,"2,457.54","2,057.24"
...,...,...,...,...,...,...,...
195,A009150,삼성전기,2025Q2,2026Q2,"2,130.08","4,403.69",106.74
196,A084010,대한제강,2025Q2,2026Q2,79.20,162.93,105.72
197,A000680,LS네트웍스,2025Q2,2026Q2,306.64,630.54,105.63
198,A000050,경방,2025Q2,2026Q2,54.92,111.66,103.32


## 4. QoQ 성장률 상위 N개

In [7]:
A.qoq_screen(engine, '매출액', n=TOP_N, asof=ASOF, market=MARKET, unit=UNIT)

,ticker,company_name,base_q,t_q,매출액(t-1),매출액(t),growth_%
0,A039200,오스코텍,2026Q1,2026Q2,36.48,526.49,"1,343.13"
1,A000040,KR모터스,2026Q1,2026Q2,25.55,263.48,931.15
2,A372910,한컴라이프케어,2026Q1,2026Q2,59.89,550.74,819.62
3,A475150,SK이터닉스,2026Q1,2026Q2,275.23,"2,408.92",775.23
4,A365270,큐라클,2026Q1,2026Q2,10.49,71.85,584.91
...,...,...,...,...,...,...,...
95,A003010,혜인,2026Q1,2026Q2,403.65,641.51,58.93
96,A446540,메가터치,2026Q1,2026Q2,158.77,252.16,58.82
97,A011330,유니켐,2026Q1,2026Q2,195.61,309.63,58.29
98,A217820,원익피앤이,2026Q1,2026Q2,272.94,431.86,58.23


In [8]:
A.qoq_screen(engine, '영업이익', n=TOP_N, asof=ASOF, market=MARKET, unit=UNIT)

,ticker,company_name,base_q,t_q,영업이익(t-1),영업이익(t),growth_%
0,A035760,CJ ENM,2026Q1,2026Q2,14.60,334.47,"2,191.43"
1,A036830,솔브레인홀딩스,2026Q1,2026Q2,14.92,318.61,"2,035.60"
2,A042700,한미반도체,2026Q1,2026Q2,84.56,"1,303.47","1,441.42"
3,A036190,금화피에스시,2026Q1,2026Q2,12.84,168.24,"1,209.77"
4,A010060,OCI홀딩스,2026Q1,2026Q2,108.62,"1,080.86",895.05
...,...,...,...,...,...,...,...
95,A093370,후성,2026Q1,2026Q2,92.62,233.99,152.64
96,A130660,한전산업,2026Q1,2026Q2,63.27,159.24,151.67
97,A143160,아이디스,2026Q1,2026Q2,34.05,85.55,151.25
98,A004150,한솔홀딩스,2026Q1,2026Q2,79.40,198.53,150.05


## 5. 영업이익 흑자전환

In [9]:
A.turnaround_screen(engine, basis='yoy', asof=ASOF, market=MARKET, unit=UNIT)   # t-4 적자 → t 흑자

,ticker,company_name,base_q,t_q,영업이익(t-4),영업이익(t),swing,매출액(t),swing_%rev
0,A034730,SK,2025Q2,2026Q2,-738.58,"48,412.47","49,151.05","421,246.76",11.67
1,A096770,SK이노베이션,2025Q2,2026Q2,"-6,909.81","34,872.96","41,782.77","291,572.05",14.33
2,A010950,S-Oil,2025Q2,2026Q2,"-3,439.71","9,650.15","13,089.86","113,434.88",11.54
3,A051910,LG화학,2025Q2,2026Q2,-139.78,"5,995.88","6,135.66","141,759.01",4.33
4,A006400,삼성SDI,2025Q2,2026Q2,"-4,642.21",961.07,"5,603.28","37,688.08",14.87
...,...,...,...,...,...,...,...,...,...
137,A000040,KR모터스,2025Q2,2026Q2,-2.33,4.35,6.69,263.48,2.54
138,A016790,현대사료,2025Q2,2026Q2,-2.54,2.96,5.50,295.25,1.86
139,A002690,동일제강,2025Q2,2026Q2,-1.46,3.94,5.40,419.42,1.29
140,A269620,시스웍,2025Q2,2026Q2,-2.96,1.06,4.02,37.31,10.76


In [10]:
A.turnaround_screen(engine, basis='qoq', asof=ASOF, market=MARKET, unit=UNIT)   # t-1 적자 → t 흑자

,ticker,company_name,base_q,t_q,영업이익(t-1),영업이익(t),swing,매출액(t),swing_%rev
0,A051910,LG화학,2026Q1,2026Q2,-496.91,"5,995.88","6,492.79","141,759.01",4.58
1,A352820,하이브,2026Q1,2026Q2,"-1,965.75","1,709.25","3,674.99","14,499.98",25.34
2,A006400,삼성SDI,2026Q1,2026Q2,"-2,360.51",961.07,"3,321.58","37,688.08",8.81
3,A017940,E1,2026Q1,2026Q2,"-1,561.89","1,178.99","2,740.87","44,028.08",6.23
4,A285130,SK케미칼,2026Q1,2026Q2,-188.89,323.49,512.39,"7,451.20",6.88
...,...,...,...,...,...,...,...,...,...
120,A448900,한국피아이엠,2026Q1,2026Q2,-7.47,1.80,9.27,105.03,8.83
121,A439580,블루엠텍,2026Q1,2026Q2,-7.43,1.82,9.25,549.67,1.68
122,A234030,싸이닉솔루션,2026Q1,2026Q2,-0.32,7.53,7.85,499.96,1.57
123,A004410,서울식품,2026Q1,2026Q2,-1.22,2.24,3.46,168.43,2.05


## 6. 재무비율 스크리너
지원: `OPM, NPM, GPM, ROE, ROA, ROIC, 부채비율, 순차입금비율, OCF_margin, FCF_margin`

- 손익 항목은 TTM 합 (`ttm=True`), BS 항목은 기초/기말 평균
- ROIC = 영업이익×(1−유효세율) / (자본 + 차입금 − 현금성자산)  평균
- 자본 ≤ 0, 매출 ≤ 0 기업은 해당 비율 NaN 처리

`compute_ratios()` 를 한 번 호출해 두고 `ratios_df=` 로 넘기면 비율마다 DB 재조회 없이 정렬만 수행.

In [11]:
ratios = A.compute_ratios(engine, asof=ASOF, market=MARKET, ttm=True, unit=UNIT)
ratios.shape

(1585, 24)

In [12]:
A.ratio_screen(engine, 'ROE',  n=TOP_N, ratios_df=ratios)

,ticker,company_name,t_q,당기순이익,자본(평균),ROE,OPM,NPM,ROA,ROIC,부채비율
0,A001470,삼부토건,2026Q2,"1,763.03",20.24,"8,711.62",24.40,172.57,76.98,NaN,181.70
1,A000660,SK하이닉스,2026Q2,"1,621,119.93","1,749,178.56",92.68,68.04,85.70,67.84,61.76,32.80
2,A458870,씨어스,2026Q2,418.03,455.59,91.75,43.52,43.07,71.38,107.65,25.65
3,A402340,SK스퀘어,2026Q2,"328,090.93","386,777.76",84.83,95.87,94.58,78.87,84.68,6.62
4,A080220,제주반도체,2026Q2,"2,608.76","3,180.83",82.02,32.21,38.76,53.09,51.63,63.58
...,...,...,...,...,...,...,...,...,...,...,...
95,A023590,다우기술,2026Q2,"17,555.44","75,833.91",23.15,6.45,5.04,1.96,4.00,"1,212.39"
96,A065710,서호전기,2026Q2,224.26,977.02,22.95,22.18,22.12,16.95,32.40,32.49
97,A092230,KPX홀딩스,2026Q2,"4,042.48","17,647.68",22.91,4.91,32.42,17.10,2.61,35.75
98,A064850,에프앤가이드,2026Q2,161.14,704.40,22.88,42.96,36.33,18.76,23.06,16.59


In [13]:
A.ratio_screen(engine, 'ROIC', n=TOP_N, ratios_df=ratios)

,ticker,company_name,t_q,영업이익,유효세율_%,투하자본(평균),ROIC,OPM,NPM,ROE,ROA,부채비율
0,A092130,이크레더블,2026Q2,159.18,22.20,22.01,562.64,30.57,25.20,31.63,23.26,36.01
1,A0009K0,에임드바이오,2026Q2,242.18,22.00,83.00,227.60,36.26,57.78,41.10,25.12,3.68
2,A388050,지투파워,2026Q2,112.16,19.15,48.82,185.76,13.55,9.03,18.86,8.86,96.20
3,A060250,NHN KCP,2026Q2,606.35,26.21,311.99,143.40,4.39,4.01,18.68,8.83,118.03
4,A028050,삼성E&A,2026Q2,"9,152.00",20.73,"5,337.17",135.93,9.50,7.22,15.81,6.99,122.37
...,...,...,...,...,...,...,...,...,...,...,...,...
95,A064850,에프앤가이드,2026Q2,190.54,15.61,697.30,23.06,42.96,36.33,22.88,18.76,16.59
96,A265740,엔에프씨,2026Q2,170.58,16.18,625.83,22.85,18.42,14.52,20.66,15.56,32.33
97,A005710,대원산업,2026Q2,468.22,17.09,"1,700.94",22.82,4.66,6.96,11.93,8.87,30.50
98,A383220,F&F,2026Q2,"5,009.09",25.55,"16,408.38",22.73,24.92,26.17,28.39,22.11,26.67


In [14]:
A.ratio_screen(engine, 'OPM',  n=TOP_N, ratios_df=ratios)

,ticker,company_name,t_q,매출액,영업이익,OPM,NPM,ROE,ROA,ROIC,부채비율
0,A402340,SK스퀘어,2026Q2,"346,905.88","332,577.17",95.87,94.58,84.83,78.87,84.68,6.62
1,A350520,이지스레지던스리츠,2026Q2,401.07,368.41,91.86,NaN,NaN,NaN,NaN,NaN
2,A395400,SK리츠,2026Q2,"2,704.46","2,404.39",88.90,NaN,NaN,NaN,NaN,NaN
3,A348950,제이알글로벌리츠,2026Q2,"1,596.92","1,243.29",77.86,NaN,NaN,NaN,NaN,NaN
4,A404990,신한서부티엔디리츠,2026Q2,892.34,666.40,74.68,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
95,A071200,인피니트헬스케어,2026Q2,"1,097.33",244.41,22.27,NaN,NaN,NaN,NaN,NaN
96,A065710,서호전기,2026Q2,"1,013.65",224.84,22.18,22.12,22.95,16.95,32.40,32.49
97,A131290,티에스이,2026Q2,"5,677.03","1,257.47",22.15,20.45,23.68,17.85,22.48,35.70
98,A396470,워트,2026Q2,204.02,45.15,22.13,24.98,7.73,7.44,36.41,5.84


In [15]:
A.ratio_screen(engine, '부채비율', n=TOP_N, ratios_df=ratios, ascending=True)   # 낮은 순

,ticker,company_name,t_q,자산(평균),자본(평균),부채비율,OPM,NPM,ROE,ROA,ROIC
0,A004770,써니전자,2026Q2,860.46,844.46,2.37,13.81,25.28,3.77,3.70,2.20
1,A002870,신풍,2026Q2,795.53,770.50,2.52,-8.54,34.89,11.26,10.91,-3.22
2,A065660,안트로젠,2026Q2,872.27,855.50,2.75,-29.40,-19.32,-1.89,-1.85,-4.53
3,A000950,전방,2026Q2,"2,371.62","2,280.63",3.55,-13.79,1.62,0.22,0.21,-2.13
4,A0009K0,에임드바이오,2026Q2,"1,536.52",939.12,3.68,36.26,57.78,41.10,25.12,227.60
...,...,...,...,...,...,...,...,...,...,...,...
95,A445180,퓨릿,2026Q2,"1,208.58","1,072.01",13.65,14.18,11.73,17.06,15.14,21.80
96,A131370,알서포트,2026Q2,"1,124.97",984.99,13.78,-0.77,-1.50,-0.71,-0.62,-0.33
97,A281820,케이씨텍,2026Q2,"6,021.49","5,348.94",14.10,19.92,18.32,16.32,14.49,26.79
98,A062040,산일전기,2026Q2,"6,598.24","5,805.33",14.24,36.05,30.59,31.06,27.33,33.86


In [ ]:
# 특정 종목 비율만 보기
ratios[ratios['ticker'].isin(['A278470', 'A000660', 'A004000'])].T